# Exploracion - Kronos

Analisis exploratorio para preparar modelado de asociacion y anomalias con foco en calidad de ventas/devoluciones.


In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

CWD = Path.cwd().resolve()
candidates = [CWD, CWD.parent, CWD / "03_modelado" / "proyecto_ml_experimentos"]
ROOT = next((p for p in candidates if (p / "src" / "postgres_loader.py").exists()), None)
if ROOT is None:
    raise RuntimeError("No se encontro la raiz de proyecto_ml_experimentos.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for m in ["src", "src.postgres_loader"]:
    if m in sys.modules:
        del sys.modules[m]

from src.postgres_loader import load_dwh_query

charts_dir = ROOT.parents[1] / "05_evidencias" / "graficas"
charts_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
kronos = load_dwh_query(
    """
    SELECT centro_costo, codigo_producto, codigo_alterno, producto, mes, anio,
           cant_venta, total_venta, cant_devolucion, total_devolucion,
           cant_neto, total_neto, flag_outlier
    FROM silver.kronos_ventas
    """
)

apriori_tx = load_dwh_query(
    """
    SELECT transaccion_id, producto, fecha
    FROM silver.apriori_transacciones
    """
)

gold_kpis = load_dwh_query(
    """
    SELECT centro_costo, producto, anio, mes, cant_venta, total_venta,
           cant_neto, total_neto, cant_devolucion, total_devolucion,
           tasa_devolucion_cant, tasa_devolucion_valor
    FROM gold.kpis_ventas
    """
)

gold_agencias = load_dwh_query(
    """
    SELECT centro_costo, total_venta, total_neto, total_devolucion,
           rentabilidad, ticket_promedio, tasa_devolucion, rentabilidad_promedio
    FROM gold.metricas_agencias
    """
)

for c in ["cant_venta", "total_venta", "cant_devolucion", "total_devolucion", "cant_neto", "total_neto"]:
    kronos[c] = pd.to_numeric(kronos[c], errors="coerce").fillna(0)

for c in ["cant_venta", "total_venta", "cant_devolucion", "total_devolucion", "cant_neto", "total_neto", "tasa_devolucion_cant", "tasa_devolucion_valor"]:
    gold_kpis[c] = pd.to_numeric(gold_kpis[c], errors="coerce").fillna(0)

for c in ["total_venta", "total_neto", "total_devolucion", "rentabilidad", "ticket_promedio", "tasa_devolucion", "rentabilidad_promedio"]:
    gold_agencias[c] = pd.to_numeric(gold_agencias[c], errors="coerce").fillna(0)

display(kronos.head())
display(apriori_tx.head())
display(gold_agencias.head())


In [ ]:
null_key = int(((kronos["centro_costo"].astype(str).str.strip() == "") | (kronos["producto"].astype(str).str.strip() == "")).sum())
dup_key = int(kronos.duplicated(subset=["centro_costo", "codigo_producto", "mes", "anio"], keep=False).sum())
neg_net = int(((kronos["cant_neto"] < 0) | (kronos["total_neto"] < 0)).sum())
dev_gt_sale = int(((kronos["cant_devolucion"] > kronos["cant_venta"]) | (kronos["total_devolucion"] > kronos["total_venta"])).sum())
outliers = int(kronos.get("flag_outlier", pd.Series([False] * len(kronos))).fillna(False).astype(bool).sum())

display(
    Markdown(
        f"""
## Calidad Silver Kronos
- Filas: **{len(kronos):,}**
- Agencias: **{kronos['centro_costo'].nunique():,}**
- Productos: **{kronos['producto'].nunique():,}**
- Nulos/vacios en claves: **{null_key:,}**
- Duplicados por (`centro_costo,codigo_producto,mes,anio`): **{dup_key:,}**
- Netos negativos: **{neg_net:,}**
- Devolucion > venta (cantidad o valor): **{dev_gt_sale:,}**
- Filas con `flag_outlier`: **{outliers:,}**
"""
    )
)

tx_uniq = apriori_tx.drop_duplicates(subset=["transaccion_id", "producto"]).shape[0]
display(
    Markdown(
        f"""
## Calidad Silver Apriori
- Filas: **{len(apriori_tx):,}**
- Transacciones: **{apriori_tx['transaccion_id'].nunique():,}**
- Productos: **{apriori_tx['producto'].nunique():,}**
- Duplicados exactos (`transaccion_id`,`producto`): **{len(apriori_tx) - tx_uniq:,}**
"""
    )
)


In [ ]:
month_map = {
    "ENERO": 1, "FEBRERO": 2, "MARZO": 3, "ABRIL": 4, "MAYO": 5, "JUNIO": 6,
    "JULIO": 7, "AGOSTO": 8, "SEPTIEMBRE": 9, "SETIEMBRE": 9, "OCTUBRE": 10,
    "NOVIEMBRE": 11, "DICIEMBRE": 12,
}

tmp = kronos.copy()
tmp["mes_num"] = tmp["mes"].astype(str).str.upper().map(month_map)
tmp["periodo"] = pd.to_datetime(
    tmp["anio"].astype("Int64").astype(str) + "-" + tmp["mes_num"].astype("Int64").astype(str).str.zfill(2) + "-01",
    errors="coerce",
)

monthly = (
    tmp.dropna(subset=["periodo"]).groupby("periodo", as_index=False)[["total_venta", "total_devolucion", "total_neto"]].sum().sort_values("periodo")
)

plt.figure(figsize=(12, 4))
plt.plot(monthly["periodo"], monthly["total_venta"], marker="o", label="Venta")
plt.plot(monthly["periodo"], monthly["total_devolucion"], marker="o", label="Devolucion")
plt.plot(monthly["periodo"], monthly["total_neto"], marker="o", label="Neto")
plt.title("Tendencia mensual Silver Kronos")
plt.legend()
plt.tight_layout()
plt.savefig(charts_dir / "eda_kronos_tendencia_mensual.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(kronos["total_venta"], bins=40, color="#2ca02c")
plt.title("Distribucion total_venta")
plt.tight_layout()
plt.savefig(charts_dir / "eda_kronos_distribucion_total_venta.png", dpi=150)
plt.show()


In [ ]:
ag = (
    kronos.groupby("centro_costo", as_index=False)
    .agg(total_venta=("total_venta", "sum"), total_devolucion=("total_devolucion", "sum"), total_neto=("total_neto", "sum"))
)
ag["tasa_dev"] = np.where(ag["total_venta"] > 0, ag["total_devolucion"] / ag["total_venta"], np.nan)

display(Markdown("## Top agencias por venta"))
display(ag.sort_values("total_venta", ascending=False).head(10))

display(Markdown("## Top agencias por tasa de devolucion"))
display(ag.sort_values("tasa_dev", ascending=False).head(10))

cmp = pd.DataFrame(
    [{
        "silver_total_venta": float(kronos["total_venta"].sum()),
        "gold_total_venta": float(gold_kpis["total_venta"].sum()),
        "silver_total_devolucion": float(kronos["total_devolucion"].sum()),
        "gold_total_devolucion": float(gold_kpis["total_devolucion"].sum()),
    }]
)
display(Markdown("## Contraste Silver vs Gold"))
display(cmp)


In [ ]:
tx_size = apriori_tx.groupby("transaccion_id")["producto"].nunique()

plt.figure(figsize=(8, 4))
sns.histplot(tx_size, bins=30, color="#9467bd")
plt.title("Productos por transaccion (apriori)")
plt.tight_layout()
plt.savefig(charts_dir / "eda_kronos_apriori_productos_por_transaccion.png", dpi=150)
plt.show()

display(
    Markdown(
        f"""
## Lectura para asociacion
- Transacciones con >=2 productos: **{int((tx_size >= 2).sum()):,}**
- Promedio productos por transaccion: **{tx_size.mean():.2f}**
- P95 productos por transaccion: **{tx_size.quantile(0.95):.2f}**
"""
    )
)


In [ ]:
riesgo = "ALTO" if (dup_key > 0 or dev_gt_sale > 0) else "BAJO"
apto_asoc = "SI" if int((tx_size >= 2).sum()) > 100 else "NO"

display(
    Markdown(
        f"""
## Conclusiones para modelado Kronos
1. Asociacion: usar `silver.apriori_transacciones` como fuente base.
2. Anomalias: usar `gold.metricas_agencias` para monitoreo y `silver.kronos_ventas` para explicabilidad.
3. Riesgo de calidad Silver: **{riesgo}**.
4. Dataset apto para asociacion multi-producto: **{apto_asoc}**.
5. Gold se usa para cierre ejecutivo y validacion de consistencia.

Graficas guardadas en: `{charts_dir}`
"""
    )
)
